In [ ]:
# Run relative to the repository root so data/, models/ and figures/ resolve.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
# 1) Imports & data
import os
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import scipy
if not hasattr(scipy, "interp"):
    scipy.interp = np.interp  # restore deprecated alias

from pycaret.classification import (
    setup, get_config, create_model, tune_model, calibrate_model,
    load_model, predict_model
)

# Your raw data (same file you trained with)
df = pd.read_csv("data/data.csv")

# ---- Feature groups (match your training) ----
preop_basic = [
    "Age ", "BMI", "Tumor size", "Tumor size.1", "Tumor size.2", "Tumor volume",
    "Last BMI on file",
    "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss",
    "Pre-op hydrocephalus (y/n)", "Pre-op intacranial pressure symptoms",
    "Pre-op headache", "Pre-op endocrine changes (y/n)",
    "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional",
    "Intraventricular extension"
]

target_col = "Permanent DI"
use_cols = [c for c in (preop_basic + ["Surgical Approach",
                                       "Resection outcome (total, subtotal, partial)",
                                      target_col]) if c in df.columns]
df = df[use_cols].copy()

# Drop rows with missing target; ensure binary
df = df[~df[target_col].isna()].copy()
print("Target distribution:")
print(df[target_col].value_counts(dropna=False))

# 2) Recreate PyCaret preprocessing (same seed/folds/imbalance fixes)
clf_setup = setup(
    data=df,
    target=target_col,
    session_id=42,
    fold=3,
    fold_strategy="stratifiedkfold",
    fix_imbalance=True,
    use_gpu=False,
    verbose=False
)

# 2b) Get transformed data & feature names in PyCaret 3
target_name = get_config("target_param")

# Full transformed dataset (DataFrame) with target column present
df_transformed = get_config("dataset_transformed").copy()

# Feature names after all encoding/scaling (drop target)
feat_names = [c for c in df_transformed.columns if c != target_name]

# Train/test splits already transformed (NumPy arrays)
X_train_t = get_config("X_train_transformed")
X_test_t  = get_config("X_test_transformed")
y_train_t = get_config("y_train_transformed")
y_test_t  = get_config("y_test_transformed")

# Wrap the transformed matrices as DataFrames to keep feature names for SHAP plots
import numpy as np, pandas as pd
X_train_df = pd.DataFrame(X_train_t, columns=feat_names).reset_index(drop=True)
X_test_df  = pd.DataFrame(X_test_t,  columns=feat_names).reset_index(drop=True)

print("Transformed feature count:", len(feat_names))
print("X_train_df shape:", X_train_df.shape, "| X_test_df shape:", X_test_df.shape)

# 3) Recreate (or load) the two models (same as before)
ridge = create_model("ridge")
ridge_cal = calibrate_model(ridge, method="isotonic")

cat = create_model("catboost")
cat_t = tune_model(cat, optimize="F1")

# 4) SHAP explainers using transformed features
import shap
import matplotlib.pyplot as plt
import os

os.makedirs("figures/xai", exist_ok=True)

# ---- CatBoost (TreeExplainer) ----
cat_est = cat_t
# If PyCaret returns a wrapper, try to unwrap; otherwise this is a no-op.
for attr in ["trained_model", "trained_model_", "model", "estimator"]:
    if hasattr(cat_est, attr):
        cat_est = getattr(cat_est, attr)

tree_explainer = shap.TreeExplainer(cat_est)
shap_vals_cat = tree_explainer.shap_values(X_train_df)
exp_val_cat = tree_explainer.expected_value

# ---- Calibrated Ridge (LinearExplainer on base estimator) ----
from sklearn.calibration import CalibratedClassifierCV

def extract_linear_base_from_calibrator(cal_model):
    """
    Returns the underlying linear classifier for SHAP LinearExplainer.
    Works across sklearn versions and CV/prefit modes.
    """
    if not isinstance(cal_model, CalibratedClassifierCV):
        return cal_model  # not calibrated -> already base

    # 1) If the model has calibrated_classifiers_ (fitted with CV), take the first
    if hasattr(cal_model, "calibrated_classifiers_") and cal_model.calibrated_classifiers_:
        cc0 = cal_model.calibrated_classifiers_[0]
        # cc0 has attributes: base_estimator, calibrator
        if hasattr(cc0, "base_estimator"):
            return cc0.base_estimator

    # 2) If using prefit=True style, estimator may exist
    if hasattr(cal_model, "estimator") and cal_model.estimator is not None:
        return cal_model.estimator

    # 3) Fallback: you can always just explain the *uncalibrated* ridge you created above
    # (SHAP explains the linear decision function; calibration is a monotonic transform)
    return ridge  # <-- the uncalibrated Ridge you created earlier


lin_base = extract_linear_base_from_calibrator(ridge_cal)



Target distribution:
Permanent DI
1.0    38
0.0    20
Name: count, dtype: int64
Transformed feature count: 18
X_train_df shape: (52, 18) | X_test_df shape: (18, 18)


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7857,0.6889,0.7778,0.8750,0.8235,0.5532,0.5594
1,0.7692,0.6667,0.7778,0.8750,0.8235,0.4935,0.5007
2,0.6923,0.7500,0.7500,0.7500,0.7500,0.3500,0.3500
Mean,0.7491,0.7019,0.7685,0.8333,0.7990,0.4656,0.4700
Std,0.0407,0.0352,0.0131,0.0589,0.0347,0.0853,0.0882


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7143,0.7333,0.8889,0.7273,0.8000,0.3171,0.3373
1,0.7692,0.6806,0.7778,0.8750,0.8235,0.4935,0.5007
2,0.6923,0.6750,0.7500,0.7500,0.7500,0.3500,0.3500
Mean,0.7253,0.6963,0.8056,0.7841,0.7912,0.3869,0.3960
Std,0.0324,0.0263,0.0600,0.0649,0.0307,0.0766,0.0742


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.5000,0.5111,0.7778,0.5833,0.6667,-0.2564,-0.3043
1,0.8462,0.6667,1.0000,0.8182,0.9000,0.5806,0.6396
2,0.7692,0.5250,0.8750,0.7778,0.8235,0.4935,0.5007
Mean,0.7051,0.5676,0.8843,0.7264,0.7967,0.2726,0.2787
Std,0.1484,0.0703,0.0910,0.1025,0.0971,0.3757,0.4161


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.6429,0.4778,0.8889,0.6667,0.7619,0.1026,0.1217
1,0.8462,0.7917,1.0000,0.8182,0.9000,0.5806,0.6396
2,0.7692,0.6375,1.0000,0.7273,0.8421,0.4507,0.5394
Mean,0.7527,0.6356,0.9630,0.7374,0.8347,0.3780,0.4336
Std,0.0838,0.1282,0.0524,0.0623,0.0566,0.2018,0.2243


Fitting 3 folds for each of 10 candidates, totalling 30 fits


In [12]:
# --- 5) KernelExplainer on the SOFT-VOTE BLEND (proba) ---

import numpy as np
import shap
import matplotlib.pyplot as plt
import os
os.makedirs("figures/xai", exist_ok=True)

# (A) Define the blended proba function on TRANSFORMED FEATURES
#     We assume both models are already fitted via PyCaret cells above.
#     If cat_est was unwrapped earlier, reuse it; otherwise keep cat_t.
cat_for_proba = cat_est if 'cat_est' in globals() else cat_t

def proba_blend(X):
    """
    X: pandas DataFrame of transformed features (same columns as X_train_df)
    Returns: 1D array of positive-class probabilities for the soft-vote blend
    """
    # Ensure DataFrame
    if isinstance(X, np.ndarray):
        X = pd.DataFrame(X, columns=X_train_df.columns)
    p_cat = cat_for_proba.predict_proba(X)[:, 1]
    p_rid = ridge_cal.predict_proba(X)[:, 1]
    return 0.5 * (p_cat + p_rid)

# (B) Choose a SMALL background for KernelExplainer (speed & stability)
#     Stratified-ish sample: take ~30 rows with a bias toward class balance if possible.
bg_size = len(X_train_df)
# try to pick balanced indices if you have y_train_t; else random sample
try:
    pos_idx = np.where(y_train_t == 1)[0]
    neg_idx = np.where(y_train_t == 0)[0]
    n_pos = min(len(pos_idx), bg_size // 2)
    n_neg = min(len(neg_idx), bg_size - n_pos)
    bg_idx = np.concatenate([
        np.random.choice(pos_idx, size=n_pos, replace=False),
        np.random.choice(neg_idx, size=n_neg, replace=False),
    ])
    background = X_train_df.iloc[bg_idx].reset_index(drop=True)
except Exception:
    background = X_train_df.sample(bg_size, random_state=42).reset_index(drop=True)

# (C) Build the KernelExplainer on the blended probability
explainer = shap.KernelExplainer(model=proba_blend, data=background)

# (D) Decide what to EXPLAIN: test set, or all rows
X_explain = pd.concat([X_train_df, X_test_df], axis=0).reset_index(drop=True)
# nsamples controls accuracy vs speed; "auto" works, or set a budget like 100 * n_features
shap_vals_kernel = explainer.shap_values(X_explain, nsamples="auto")
expected_val_kernel = explainer.expected_value

# (E) Save global plots
plt.figure()
shap.summary_plot(shap_vals_kernel, X_explain, show=False)  # beeswarm
plt.tight_layout()
plt.savefig("figures/xai/blend_kernel_summary_beeswarm.png", dpi=200)
plt.close()

plt.figure()
shap.summary_plot(shap_vals_kernel, X_explain, plot_type="bar", show=False)  # mean |SHAP|
plt.tight_layout()
plt.savefig("figures/xai/blend_kernel_summary_bar.png", dpi=200)
plt.close()

# (F) A few dependence plots for top-k features
mean_abs = np.abs(shap_vals_kernel).mean(axis=0)
top_idx = np.argsort(-mean_abs)[:8]
top_feats = [X_explain.columns[i] for i in top_idx]

for fname in top_feats:
    plt.figure()
    shap.dependence_plot(
        ind=fname,
        shap_values=shap_vals_kernel,
        features=X_explain,
        interaction_index=None,
        show=False
    )
    plt.tight_layout()
    plt.savefig(f"figures/xai/blend_kernel_dependence_{fname.replace(' ','_').replace('/','_')}.png", dpi=200)
    plt.close()

# (G) Local force plots (HTML) for a few held-out samples (pick from X_test_df region)
start = len(X_train_df)
indices = list(range(start, min(start + 5, len(X_explain))))

for i in indices:
    fp = shap.force_plot(expected_val_kernel, shap_vals_kernel[i, :], X_explain.iloc[i, :], matplotlib=False)
    shap.save_html(f"figures/xai/force_blend_kernel_{i}.html", fp)

print("Kernel SHAP finished. Plots saved to ./xai_plots")


  0%|          | 0/70 [00:00<?, ?it/s]

Kernel SHAP finished. Plots saved to ./xai_plots


<Figure size 800x550 with 0 Axes>

<Figure size 800x550 with 0 Axes>

<Figure size 800x550 with 0 Axes>

<Figure size 800x550 with 0 Axes>

<Figure size 800x550 with 0 Axes>

<Figure size 800x550 with 0 Axes>

<Figure size 800x550 with 0 Axes>

<Figure size 800x550 with 0 Axes>

In [44]:
import re
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd

def _norm(s: str) -> str:
    """Normalize a feature name for robust matching."""
    return re.sub(r"[\W_]+", " ", str(s)).strip().lower()

def _ohe_group_cols(base_name: str, cols: list[str]) -> list[str]:
    """
    Return all one-hot columns belonging to a base categorical feature.
    Matching is tolerant to punctuation/spacing.
    """
    base_n = _norm(base_name)
    group = []
    for c in cols:
        # split "Feature_level" into ("Feature", "level") but allow weird chars
        if "_" in c:
            left = c.rsplit("_", 1)[0]
            if _norm(left) == base_n:
                group.append(c)
    return group

def _reconstruct_category_from_ohe(base_name: str, X: pd.DataFrame) -> pd.Series:
    """
    From one-hot columns of a base feature, reconstruct a single categorical Series.
    Picks the level with the highest indicator per row (argmax).
    If a row has all zeros for this group, returns NaN for that row.
    """
    cols = _ohe_group_cols(base_name, list(X.columns))
    if not cols:
        # Already present as a single (non-OHE) column
        if base_name in X.columns:
            return X[base_name].astype("category")
        # Nothing to reconstruct
        return pd.Series([np.nan] * len(X), index=X.index, name=base_name, dtype="category")

    levels = [c.rsplit("_", 1)[1] for c in cols]
    M = X[cols].to_numpy()

    # rows with any signal
    row_sums = M.sum(axis=1)
    best = M.argmax(axis=1)

    # default labels by argmax
    labels = np.array(levels, dtype=object)[best]

    # if a row has all zeros across this group, set NaN
    labels = pd.Series(labels, index=X.index, name=base_name)
    labels[row_sums == 0] = np.nan

    # return as categorical
    return pd.Categorical(labels, categories=levels)

def _ensure_feature_series(name: str | None, X: pd.DataFrame) -> pd.Series | None:
    """Return a Series for plotting given a requested feature that may be OHE."""
    if name is None:
        return None
    if name in X.columns:
        return X[name]
    # Try reconstruct from OHE
    s = _reconstruct_category_from_ohe(name, X)
    if s.isna().all():
        # Try looser match: find a column with same normalized name
        nm = _norm(name)
        for c in X.columns:
            if _norm(c) == nm:
                return X[c]
        raise AssertionError(f"{name} not found in X (nor reconstructable from one-hot).")
    return s

def _group_shap_for_feature(name: str, shap_df: pd.DataFrame, X: pd.DataFrame) -> pd.Series:
    """
    Return SHAP contributions for a requested feature.
    - If it’s a single numeric/continuous column: that column’s SHAP.
    - If it’s a base categorical with OHE: sum SHAP across its OHE columns.
    """
    if name in shap_df.columns:
        return shap_df[name]
    group_cols = _ohe_group_cols(name, list(shap_df.columns))
    if group_cols:
        return shap_df[group_cols].sum(axis=1)
    # Last resort: normalized exact match
    nm = _norm(name)
    for c in shap_df.columns:
        if _norm(c) == nm:
            return shap_df[c]
    raise AssertionError(f"Could not map {name} to SHAP columns.")

def make_shap_dependence_plot(
    shap_values: np.ndarray,
    X: pd.DataFrame,
    feature_x: str,
    feature_color: str | None = None,
    feature_size: str | None = None,
    feature_symbol: str | None = None,
    title: str | None = None,
    sample: int | None = None,
    smooth: bool = True,
    bins: int = 20,
    y_mode: str = "by_x",  # "by_x" -> SHAP of feature_x; "sum" -> total SHAP per row
):
    """
    Interactive SHAP dependence plot (Plotly).
    y-axis = SHAP value (Permanent DI).
      - y_mode="by_x": SHAP contribution of feature_x (or sum over its OHE group)
      - y_mode="sum":  total SHAP across all features

    Accepts either original categorical names or encoded OHE column names for
    feature_x / feature_color / feature_size / feature_symbol.
    """
    # Align SHAP → DataFrame
    if shap_values.ndim == 1:
        shap_df = pd.DataFrame({X.columns[0]: shap_values})
    else:
        shap_df = pd.DataFrame(shap_values, columns=X.columns)

    # Build plotting frame
    dfp = X.copy().reset_index(drop=True)

    # X-axis series (handles OHE categories)
    x_series = _ensure_feature_series(feature_x, dfp)

    # y-axis series
    if y_mode == "sum":
        y_series = shap_df.sum(axis=1)
    else:
        y_series = _group_shap_for_feature(feature_x, shap_df, dfp)

    # Optional subsample
    if sample is not None and sample < len(dfp):
        idx = dfp.sample(sample, random_state=42).index
        dfp = dfp.loc[idx]
        x_series = x_series.loc[idx]
        y_series = y_series.loc[idx]

    plot_df = pd.DataFrame({"__x__": x_series, "SHAP": y_series}, index=dfp.index)

    # Aux encodings
    color_s  = _ensure_feature_series(feature_color, dfp) if feature_color else None
    size_s   = _ensure_feature_series(feature_size, dfp)  if feature_size else None
    symbol_s = _ensure_feature_series(feature_symbol, dfp) if feature_symbol else None

    if color_s is not None:  plot_df["__color__"] = color_s
    if symbol_s is not None: plot_df["__symbol__"] = symbol_s

    # Size must be numeric; if categorical, map to codes
    if size_s is not None:
        if not pd.api.types.is_numeric_dtype(size_s):
            plot_df["__size__"] = pd.Categorical(size_s).codes + 1
        else:
            plot_df["__size__"] = size_s

    # Title
    if title is None:
        title = f"SHAP dependence: {feature_x} → SHAP (Permanent DI)"

    # -------- PLOTTING --------
    # Build hover data and nice labels (hide __color__ everywhere)
    hover_keys = [c for c in ["__x__", "SHAP", "__size__", "__symbol__"] if c in plot_df.columns]
    labels_map = {
        "__x__": feature_x,
        "SHAP": "SHAP value (Permanent DI)",
        "__size__": feature_size if feature_size else "",
        "__symbol__": feature_symbol if feature_symbol else "",
        "__color__": "",  # suppress label in legend/colorbar
    }

    fig = px.scatter(
        plot_df,
        x="__x__", y="SHAP",
        color="__color__" if "__color__" in plot_df.columns else None,
        size="__size__"  if "__size__"  in plot_df.columns else None,
        symbol="__symbol__" if "__symbol__" in plot_df.columns else None,
        hover_data=hover_keys,       # do not include __color__ -> no "__color__" in hover
        labels=labels_map,           # rename axis/legend labels
        title=title,
        template="plotly_white",
        opacity=0.85,
    )

    # Remove colorbar title if continuous color sneaks in
    fig.update_layout(coloraxis_colorbar=dict(title=""))

    # y=0 reference
    fig.add_hline(y=0, line_dash="dot", line_width=1, line_color="gray")

    # Smooth curve if x is numeric — keep it out of legend (no obstruction)
    if smooth and pd.api.types.is_numeric_dtype(plot_df["__x__"]):
        x_vals = plot_df["__x__"].to_numpy()
        y_vals = plot_df["SHAP"].to_numpy()
        q = np.linspace(0, 1, bins + 1)
        edges = np.unique(np.quantile(x_vals, q))
        x_mid, y_med = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            sel = (x_vals >= lo) & (x_vals <= hi)
            if sel.sum() >= 3:
                x_mid.append(np.median(x_vals[sel]))
                y_med.append(np.median(y_vals[sel]))
        if len(x_mid) >= 2:
            fig.add_trace(
                go.Scatter(
                    x=x_mid, y=y_med,
                    mode="lines+markers",
                    name="Smoothed median",
                    showlegend=False,          # <--- hide legend entry
                    line=dict(width=3),
                    marker=dict(size=6),
                    hovertemplate="x=%{x:.3g}<br>median SHAP=%{y:.3g}<extra></extra>",
                )
            )
    else:
        fig.update_xaxes(type="category")

    # Layout / margins; keep legend tidy and out of the way
    fig.update_layout(
        xaxis_title=feature_x,
        yaxis_title="SHAP value (Permanent DI)",
        legend_title_text="",              # no "__color__" legend title
        legend=dict(orientation="h", y=1.02, x=0),  # top outside
        margin=dict(l=70, r=90, t=60, b=60),
    )
    return fig


In [31]:
X_explain.columns

Index(['Age ', 'BMI', 'Tumor size', 'Tumor size.1', 'Tumor size.2',
       'Tumor volume', 'Last BMI on file',
       'Pre-op visual impairment (y/n) - symptomatic complaint of visual loss',
       'Pre-op hydrocephalus (y/n)', 'Pre-op intacranial pressure symptoms',
       'Pre-op headache', 'Pre-op endocrine changes (y/n)',
       'Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional',
       'Intraventricular extension', 'Surgical Approach',
       'Resection outcome (total subtotal partial)_total',
       'Resection outcome (total subtotal partial)_subtotal',
       'Resection outcome (total subtotal partial)_near total'],
      dtype='object')

In [16]:
df.columns

Index(['Age ', 'BMI', 'Tumor size', 'Tumor size.1', 'Tumor size.2',
       'Tumor volume', 'Last BMI on file',
       'Pre-op visual impairment (y/n) - symptomatic complaint of visual loss',
       'Pre-op hydrocephalus (y/n)', 'Pre-op intacranial pressure symptoms',
       'Pre-op headache', 'Pre-op endocrine changes (y/n)',
       'Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional',
       'Intraventricular extension', 'Surgical Approach',
       'Resection outcome (total, subtotal, partial)', 'Permanent DI'],
      dtype='object')

In [46]:
# 2-way
fig1 = make_shap_dependence_plot(shap_vals_kernel, X_explain, feature_x="Tumor size")
fig1.show()

# 3-way
fig2 = make_shap_dependence_plot(
    shap_vals_kernel, X_explain,
    feature_x="BMI",
    feature_color="Surgical Approach",
    feature_size="Tumor volume",
    title="BMI vs SHAP with Approach color & Volume size",
)
fig2.show()

# OHE base name now OK
fig3 = make_shap_dependence_plot(
    shap_vals_kernel, X_explain,
    feature_x="Resection outcome (total, subtotal, partial)",
    feature_color="Tumor volume",
    feature_symbol="Surgical Approach",
    title="Resection outcome vs SHAP with headache color & approach symbol",
)
fig3.show()
